In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import torch.nn as nn
from transformers import AutoTokenizer
import torch

dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
tokenizer_fp = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

class Evaluator:
    def __init__(self, dataset, tokenizer, device, n_samples=5):
        self.dataset = tokenizer(
            "\n\n".join(dataset["text"]),
            return_tensors="pt"
        ).input_ids.to(device)

        self.device = device
        self.n_samples = n_samples
        self.seq_len = 512

    @torch.no_grad()
    def evaluate(self, model):
        model.eval()
        # nlls = []
        total_loss = 0
        total_tokens = 0
        loss_fct = nn.CrossEntropyLoss()

        for i in tqdm(range(self.n_samples)):
            batch = self.dataset[:, i*self.seq_len:(i+1)*self.seq_len]
            
            outputs = model(batch)
            logits = outputs.logits
            print(outputs.logits.abs().max())
            shift_logits = logits[:, :-1, :].float()
            shift_labels = batch[:, 1:]

            loss = loss_fct(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1)
            )
            print("loss :",loss.item())
            num_tokens = shift_labels.numel()

            total_loss += loss.item() * num_tokens
            total_tokens += num_tokens

        ppl = torch.exp(torch.tensor(total_loss / total_tokens))
            # nlls.append(loss * self.seq_len)

        # ppl = torch.exp(torch.stack(nlls).sum() / (self.n_samples * self.seq_len))
        return ppl.item()

In [2]:
from transformers import AutoModelForCausalLM
model_fp=AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b-hf",torch_dtype=torch.float16,device_map="auto")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:31<00:00,  9.23it/s]
Some parameters are on the meta device because they were offloaded to the cpu.


In [1]:
evaluator=Evaluator(dataset,tokenizer_fp,model_fp.device)

NameError: name 'Evaluator' is not defined

In [26]:
fp_ppl=evaluator.evaluate(model_fp)

 20%|██        | 1/5 [00:04<00:18,  4.67s/it]

tensor(27.7500, device='cuda:0', dtype=torch.float16)
loss : 1.6777775287628174


 40%|████      | 2/5 [00:09<00:14,  4.73s/it]

tensor(34.2500, device='cuda:0', dtype=torch.float16)
loss : 1.961720585823059


 60%|██████    | 3/5 [00:14<00:09,  4.91s/it]

tensor(29.4219, device='cuda:0', dtype=torch.float16)
loss : 2.074246883392334


 80%|████████  | 4/5 [00:19<00:05,  5.09s/it]

tensor(31.1719, device='cuda:0', dtype=torch.float16)
loss : 1.7974156141281128


100%|██████████| 5/5 [00:24<00:00,  4.91s/it]

tensor(34.7188, device='cuda:0', dtype=torch.float16)
loss : 2.106584310531616


In [27]:
fp_ppl

6.845208644866943

In [9]:
model_quantized_path='../save'
model_quantized=AutoModelForCausalLM.from_pretrained(model_quantized_path,torch_dtype=torch.float16,device_map="auto")

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 628.22it/s]
LlamaForCausalLM LOAD REPORT from: ../save
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.layers.{0...31}.self_attn_q_proj_smooth_rotate           | UNEXPECTED |  | 
model.layers.{0...31}.input_layernorm.bias                     | UNEXPECTED |  | 
model.layers.{0...31}.mlp.up_proj.bias                         | UNEXPECTED |  | 
model.layers.{0...31}.mlp.gate_proj.weight_quantizer.zeros     | UNEXPECTED |  | 
model.layers.{0...31}.mlp.gate_proj.bias                       | UNEXPECTED |  | 
model.layers.{0...31}.post_attention_layernorm.bias            | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.q_proj.weight_quantizer.scales | UNEXPECTED |  | 
model.layers.{0...31}.self_attn_o_proj_smooth_rotate           | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.q_proj.weight_quantizer.zeros  | UNEX

In [11]:
tokenizer_quantized=AutoTokenizer.from_pretrained(model_quantized_path)

In [28]:
evaluator_quant=Evaluator(dataset,tokenizer_fp,model_quantized.device)

In [29]:
quant_ppl=evaluator_quant.evaluate(model_quantized)

 20%|██        | 1/5 [00:07<00:30,  7.64s/it]

tensor(9.0703, device='cuda:0', dtype=torch.float16)
loss : 14.12023639678955


 40%|████      | 2/5 [00:16<00:24,  8.18s/it]

tensor(10.1641, device='cuda:0', dtype=torch.float16)
loss : 12.860118865966797


 60%|██████    | 3/5 [00:24<00:16,  8.35s/it]

tensor(10.0391, device='cuda:0', dtype=torch.float16)
loss : 13.062597274780273


 80%|████████  | 4/5 [00:33<00:08,  8.45s/it]

tensor(9.7344, device='cuda:0', dtype=torch.float16)
loss : 11.693229675292969


100%|██████████| 5/5 [00:41<00:00,  8.39s/it]

tensor(10.2656, device='cuda:0', dtype=torch.float16)
loss : 12.34760856628418


In [30]:
quant_ppl

368338.6875

In [13]:
text = "Hello world"

inputs = tokenizer_fp(text, return_tensors="pt").to(model_quantized.device)

out = model_quantized(**inputs)

print(out.logits.mean())
print(out.logits.std())

tensor(0.3279, device='cuda:0', dtype=torch.float16, grad_fn=<MeanBackward0>)
tensor(2.0547, device='cuda:0', dtype=torch.float16, grad_fn=<StdBackward0>)


In [32]:
text = "The capital of France is"

inputs = tokenizer_fp(text, return_tensors="pt").to(model_quantized.device)

fp = model_fp(**inputs).logits
qt = model_quantized(**inputs).logits

print("FP logits std :", fp.std())
print("Quant logits std :", qt.std())

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 42.45 GiB is allocated by PyTorch, and 41.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)